In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("./dataset/machine_failure.csv")
df

,id,type,temp_air,temp_process,speed,torque,tool_wear,is_failure
0,14860,M,298.1,308.6,1551,42.8,0,0
1,47181,L,298.2,308.7,1408,46.3,3,0
2,47182,L,298.1,308.5,1498,49.4,5,0
3,47183,L,298.2,308.6,1433,39.5,7,0
4,47184,L,298.2,308.7,1408,40.0,9,0
...,...,...,...,...,...,...,...,...
9977,24855,M,298.8,308.4,1604,29.5,14,0
9978,39410,H,298.9,308.4,1632,31.8,17,0
9979,24857,M,299.0,308.6,1645,33.4,22,0
9980,39412,H,299.0,308.7,1408,48.5,25,0


In [56]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, train_size=0.7, random_state=123, stratify=df["type"])
len(df_train), len(df_test)

(6987, 2995)

In [19]:
from sklearn.preprocessing import StandardScaler

drop_columns=["id", "type", "is_failure"]
df_train_num = df_train.drop(columns=drop_columns)
df_test_num = df_test.drop(columns=drop_columns)

scaler = StandardScaler().fit(df_train_num)

In [20]:
arr_train = scaler.transform(df_train_num)
arr_test = scaler.transform(df_test_num)

In [22]:
df_train_norm = pd.DataFrame(arr_train, columns=df_train_num.columns)
df_test_norm = pd.DataFrame(arr_test, columns=df_test_num.columns)

In [30]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder().fit(df_train[["type"]])

In [33]:
arr_train = encoder.transform(df_train[["type"]]).toarray()
arr_test = encoder.transform(df_test[["type"]]).toarray()

In [43]:
df_train_ohe = pd.DataFrame(arr_train, columns=encoder.get_feature_names_out(input_features=["type"]))
df_test_ohe = pd.DataFrame(arr_test, columns=encoder.get_feature_names_out(input_features=["type"]))

In [57]:
df_train_fin = pd.concat([df_train[["id", "is_failure"]].reset_index(drop=True), df_train_ohe, df_train_norm], axis=1)
df_test_fin = pd.concat([df_test[["id", "is_failure"]].reset_index(drop=True), df_test_ohe, df_test_norm], axis=1)

In [62]:
df_train_fin["xgrp"] = "train"
df_test_fin["xgrp"] = "test"
df_bind = pd.concat([df_train_fin, df_test_fin]).reset_index(drop=True)

In [63]:
df_bind

,id,is_failure,type_H,type_L,type_M,temp_air,temp_process,speed,torque,tool_wear,xgrp
0,55100,0,0.0,1.0,0.0,0.399154,1.352290,-0.877336,0.926149,1.084815,train
1,49876,0,0.0,1.0,0.0,0.001026,-0.266219,-0.482743,0.596147,-0.889909,train
2,37244,0,1.0,0.0,0.0,-0.048740,0.812787,-0.240351,-0.423858,1.053470,train
3,52001,0,0.0,1.0,0.0,1.742836,1.284852,-0.747684,0.556147,-1.532478,train
4,55373,0,0.0,1.0,0.0,-0.247804,0.543036,1.890449,-1.773865,1.617677,train
...,...,...,...,...,...,...,...,...,...,...,...
9977,20233,0,0.0,0.0,1.0,1.444240,1.554604,1.529679,-1.503863,1.131832,test
9978,49442,0,0.0,1.0,0.0,-0.446868,-1.142911,-0.026143,0.236145,-0.936926,test
9979,56727,0,0.0,1.0,0.0,-0.198038,0.745349,0.796864,-0.933860,0.332539,test
9980,56130,0,0.0,1.0,0.0,-1.292891,-1.210349,-0.668765,0.996149,-0.874237,test


In [71]:
df_train = df_bind.loc[df_bind["xgrp"] == "train", ].drop(columns=["id", "xgrp"]).reset_index(drop=True)
df_test = df_bind.loc[df_bind["xgrp"] == "test", ].drop(columns=["id", "xgrp"]).reset_index(drop=True)

In [74]:
from sklearn.linear_model import LogisticRegression

In [75]:
model = LogisticRegression(random_state=123)
model.fit(X = df_train.drop(columns=["is_failure"]), y=df_train["is_failure"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,123
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [79]:
arr_pred = model.predict(X=df_test.drop(columns=["is_failure"]))
arr_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(2995,))

In [89]:
arr_pred_proba = model.predict_proba(df_test.drop(columns=["is_failure"]))
arr_pred_proba

array([[0.99444914, 0.00555086],
       [0.99165711, 0.00834289],
       [0.95139568, 0.04860432],
       ...,
       [0.9988216 , 0.0011784 ],
       [0.99174463, 0.00825537],
       [0.99434147, 0.00565853]], shape=(2995, 2))

In [92]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [93]:
roc_auc_score(df_test["is_failure"], arr_pred_proba[:, 1])

0.9109339899429996

In [94]:
accuracy_score(y_true=df_test["is_failure"], y_pred=arr_pred)

0.9719532554257095

In [95]:
f1_score(y_true=df_test["is_failure"], y_pred=arr_pred)

0.3114754098360656